# Wiki Scrape

a script to scrape star names off of wikipedia's list of stars, therefore assuring that data on pageviews can be found, and clean the data

In [2]:
## imports

import requests
import random
import pandas as pd
from io import StringIO
from astroquery.simbad import Simbad
import time

# Data Extraction

In [3]:
## constants

HEADERS = {'User-Agent': 'CelestialCelebs (https://github.com/etu8924/CelestialCelebs)'}

In [4]:
## a function to save html data of a wikipedia page with wikipedia api
#
# params:
#   page: the name of the Wikipedia page to fetch data for
#   file_path: the path to the file where the fetched data will be saved
# returns:
#   None

def get_wiki_data(page, file_path):
    r = requests.get(
        'https://en.wikipedia.org/w/api.php',
        params={'action': 'parse', 'format': 'json', 'page': page, 'prop': 'text', 'formatversion': 2},
        headers=HEADERS,
        timeout=60
    )
    r.raise_for_status()
    data = r.json()['parse']['text']

    # write to file so we can inspect data without repeatedly scraping
    f = open(file_path, 'w')
    f.write(data)
    f.close()

In [5]:
## get wiki data for stars, exoplanets, and galaxies
# uncomment if stars.html, exoplanets.html, or galaxies.html are not present in the repo

# get_wiki_data('List_of_proper_names_of_stars', 'stars.html')
get_wiki_data('List of planetary nebulae', 'nebulae.html')

# Data Parsing and Cleaning

In [6]:
## constants

# greek letter replacements
GREEK_REPLACEMENTS = {
    'α': 'alf',
    'β': 'bet',
    'η': 'eta',
    'θ': 'tet',
    'ο': 'omi',
    'τ': 'tau',
    'υ': 'ups',
    'γ': 'gam',
    'κ': 'kap',
    'λ': 'lam',
    'μ': 'mu',
    'ν': 'nu',
    'ξ': 'xi',
    'ρ': 'rho',
    'σ': 'sig',
    'φ': 'phi',
    'χ': 'chi',
    'ψ': 'psi',
    'ω': 'ome',
    'ε': 'eps',
    'δ': 'del',
    'ζ': 'zet',
    'ι': 'iota',
    'π': 'pi',
    'ς': 'sig'
}

# replacements for non-english characters in common names and star designations
CHAR_REPLACEMENTS = {
    'à': 'a', 'á': 'a', 'â': 'a', 'ä': 'a', 'ǎ': 'a', 'æ': 'ae', 'ã': 'a', 'å': 'a', 'ā': 'a',
    'è': 'e', 'é': 'e', 'ê': 'e', 'ë': 'e', 'ě': 'e', 'ẽ': 'e', 'ē': 'e', 'ė': 'e', 'ę': 'e',
    'ì': 'i', 'í': 'i', 'î': 'i', 'ï': 'i', 'ǐ': 'i', 'ĩ': 'i', 'ī': 'i', 'ı': 'i', 'į': 'i',
    'ò': 'o', 'ò': 'o', 'ó': 'o', 'ô': 'o', 'ö': 'o', 'ǒ': 'o', 'œ': 'oe', 'ø': 'o', 'õ': 'o', 'ō': 'o', 'ö̀': 'o',
    'ù': 'u', 'ú': 'u', 'û': 'u', 'ü': 'u', 'ǔ': 'u', 'ũ': 'u', 'ū': 'u', 'ű': 'u', 'ů': 'u', 'ũ̀': 'u', 'ù': 'u',
    'ş': 's', 'ș': 's', 'ś': 's', 'š': 's',
    'ç': 'c', 'ć': 'c', 'č': 'c', 'ċ': 'c',
    'ý': 'y', 'ŷ': 'y', 'ÿ': 'y', 'ỹ': 'y',
    'ñ': 'n', 'ņ': 'n', 'ň': 'n',
    'ġ': 'g', 'ğ': 'g',
    'ľ': 'l', 'ł': 'l',
    '†': '', "'": '', "ʼ": ''
}

In [7]:
## a function to parse html data, determine linked entries, and clean text
#
# params:
#   html_file: path to html file of data
#   table_match: a column header in the table of interest, unique to the table
#   cols: columns of interest
#   link_cols: columns where the link is of interest
#   designation: column of designated names, if applicable
# returns:
#    pandas dataframe of parsed data

def parse_data (html_file, table_match, cols, link_cols, designation=''):
    obj_data = open(html_file, 'r').read()

    # clean english
    for char in CHAR_REPLACEMENTS.keys():
        obj_data = obj_data.replace(char, CHAR_REPLACEMENTS[char])
        obj_data = obj_data.replace(char.upper(), CHAR_REPLACEMENTS[char].upper())

    obj_df = pd.read_html(StringIO(obj_data), match=table_match, extract_links='body')[0][cols] # read first matching table and cols
    obj_df['page_name'] = '' # eventually store the link

    # clean up linked entries in the data frame
    for r in range(0, obj_df.shape[0]):
        row = obj_df.loc[r]
        for c in link_cols:
            # check that the entry is linked, and that the link is a valid wikipedia page
            if ((type(row[c]) == tuple) and (row[c][1] is not None) and (row[c][1][:6] == '/wiki/')):
                link = row[c][1][6:]
                if ('#' in link): # doesn't have its own designated page - drop
                    link = ''
                obj_df.loc[r,'page_name'] = link
        for c in cols:
            if (type(row[c]) == tuple):
                obj_df.loc[r,c] = row[c][0]

    # handle designations
    if (designation != ''):
        for i in range(obj_df.shape[0]):
            # greek character if applicable
            d = obj_df.loc[i, designation]
            if (ord(d[0]) >= 128):
                d = d.replace(d[0], GREEK_REPLACEMENTS[d[0]])

            # multiple-star systems
            words = d.split(' ')
            if ((words[-1] == 'A') or (words[-1] == 'Aa') or (words[-1] == 'Aa1')):
                d = ' '.join(words[:-1])

            obj_df.loc[i, designation] = d

    # drop unlinked entries
    for i in range(obj_df.shape[0]):
        if (obj_df.loc[i,'page_name'] == ''):
            obj_df.drop(index=i)
    obj_df.reset_index(drop=True)

    return obj_df

In [8]:
## a function to get coordinate data of objects in the dataframe
#
# params:
#   obj_df: dataframe of objects
#   designation: name of the column with object designations
# returns:
#   dataframe of objects with SIMBAD id, RA, and dec columns

def get_loc_data (obj_df, designation):
    astroquery_df = Simbad.query_objects(obj_df[designation])['main_id', 'ra', 'dec', 'user_specified_id'].to_pandas()
    astroquery_df.rename(columns={'user_specified_id': designation}, inplace=True)
    data_df= pd.merge(obj_df, astroquery_df, on=designation)
    return data_df

## Stars

In [ ]:
## obtain star data!

stars = parse_data('stars.html', 'Modern proper name', ['Modern proper name', 'Designation'], ['Modern proper name', 'Designation'], designation='Designation')
stars_data = get_loc_data(stars, 'Designation')
# stars_data = stars_data.rename(columns={'Modern proper name':'modern_name', 'Designation':'designation'})
stars_data.to_csv('star_data.csv', columns=['modern_name', 'designation', 'page_name', 'main_id', 'ra', 'dec'], index=False)

## Exoplanets

In [ ]:
## obtain exoplanet data!

exoplanets = parse_data('exoplanets.html', 'Name of planet', ['Name of planet', 'Name of star'], ['Name of planet'])
exoplanets_data = get_loc_data(exoplanets, 'Name of planet')
# exoplanets_data = exoplanets_data.rename(columns={'Name of planet':'planet_name', 'Name of star':'star_name'})
exoplanets_data.to_csv('exoplanets_data.csv', columns=['planet_name', 'star_name', 'page_name', 'main_id', 'ra', 'dec'], index=False)

# Nebulae

In [102]:
nebulae_data = open('nebulae.html', 'r').read()
nebulae_df = pd.read_html(StringIO(nebulae_data), match='Name', extract_links='body')[0][['Image', 'Name', 'Messier Catalogue[a]', 'NGC[a]', 'Other designation[a]']]
nebulae_df = nebulae_df.rename(columns={'Image':'img','Name':'name','Messier Catalogue[a]':'messier','NGC[a]':'ngc','Other designation[a]':'other'})

In [ ]:
## the nebulae table looks a bit different than the others ... new code!

nebulae_df['designation'] = ''
nebulae_df['page_name'] = ''

# clean links
for r in range(0, nebulae_df.shape[0]):
    row = nebulae_df.loc[r]
    for c in ['name', 'messier', 'ngc', 'other']:
        # link
        if ((type(row[c]) == tuple) and (row[c][1] is not None) and (row[c][1][:6] == '/wiki/')):
            link = row[c][1][6:]
            if ('#' in link): # doesn't have its own designated page - drop
                link = ''
            nebulae_df.loc[r,'page_name'] = link
    # image link
    for c in ['name', 'messier', 'ngc', 'other']:
        if (type(row[c]) == tuple):
            nebulae_df.loc[r,c] = row[c][0]
    # clean all other links
    if (type(row['img']) == tuple):
        nebulae_df.loc[r,'img'] = row['img'][1]

# get designation in one column
for r in range(0, nebulae_df.shape[0]):
    row = nebulae_df.loc[r]
    for c in ['messier', 'ngc', 'other']:
        if ((row[c] != '') and (nebulae_df.loc[r,'designation'] == '')):
            nebulae_df.loc[r,'designation'] = row[c]

nebulae_df.loc[123,'designation'] = nebulae_df.loc[123,'designation'].replace('-', ' ')

# ra and dec data
astroquery_df = Simbad.query_objects(nebulae_df['designation'])['main_id', 'ra', 'dec', 'user_specified_id'].to_pandas()
astroquery_df = astroquery_df.rename(columns={'user_specified_id':'designation'})
nebulae_data_df = pd.merge(nebulae_df, astroquery_df, on='designation')
nebulae_data_df.dropna(inplace=True)
nebulae_data_df.reset_index(drop=True, inplace=True)

# save as a csv
nebulae_data_df.to_csv('../data/nebulae_data.csv', index=False)

# WIP
EVERYHTING BELOW IS A WIP

## Galaxies

main useful info: name

In [71]:
## read data

galaxy_data = open('galaxies.html', 'r').read()

In [72]:
## parse html data and fix exoplanet names and designations

galaxies_df = pd.read_html(StringIO(galaxy_data), match='Galaxy', extract_links='body')[0] # read first table matching "galaxy"
N_GALAXIES = galaxies_df.shape[0] # total number of rows
galaxies_df['Linked'] = False # by default assume there's no link

for r in range(1, N_GALAXIES):
    row = galaxies_df.loc[r]
    
    if ((row['Galaxy'][1] is not None) and
        (row['Galaxy'][1][:6] == '/wiki/')):
        # if the name is linked, the entry is linked
        galaxies_df.loc[r, 'Linked'] = True
    galaxies_df.loc[r, 'Galaxy'] = row['Galaxy'][0] # fix name

In [73]:
## a function to randomly select galaxies from the table, given that the galaxy has a valid hyperlink to another page
#
# params:
#   n: number of random exoplanets to select
# returns:
#   list containing [official name] for each selected exoplanet

def select_random_galaxies(n):
    objs = []
    row_idxs = random.sample(range(1, N_GALAXIES), n)
    for idx in row_idxs:
        # get row from table
        row = galaxies_df.loc[idx]
        if (row['Linked']):
            objs.append([row['Galaxy']])
        
    return pd.DataFrame(objs, columns=['official_name'])

In [ ]:
## select 75 galaxies and save as a csv
select_random_galaxies(50).to_csv('galaxies.csv', index=False)

## Asteroids

main useful info: name

In [36]:
## read data

asteroids = open('asteroids.html', 'r').read()

In [47]:
## parse html data and fix asteroid names

asteroids_df = pd.read_html(StringIO(asteroids), match='Name', extract_links='body')[0] # read first table matching "name"
N_ASTEROIDS = asteroids_df.shape[0] # total number of rows
asteroids_df['Linked'] = False # by default assume there's no link

for r in range(0, N_ASTEROIDS):
    row = asteroids_df.loc[r]
    
    if ((row['Name'][1] is not None) and
        (row['Name'][1][:6] == '/wiki/')):
        # if the name is linked, the entry is linked
        asteroids_df.loc[r, 'Linked'] = True
    asteroids_df.loc[r, 'Name'] = row['Name'][0] # fix name

# remove irrelevant rows - see original wikipedia page for details
asteroids_df = asteroids_df[:40]
N_ASTEROIDS = asteroids_df.shape[0] # total number of rows

In [53]:
## a function to return all asteroids that have a valid hyperlink to another page
#
# params:
#   n: number of random asteroids to select
# returns:
#   list containing [official name] for each selected asteroid

def select_asteroids(n):
    objs = []
    for idx in range(0, N_ASTEROIDS):
        # get row from table
        row = asteroids_df.loc[idx]
        if (row['Linked']):
            objs.append([row['Name']])
        
    return pd.DataFrame(objs, columns=['official_name'])

In [54]:
## select the 40 asteroids and save as a csv
select_asteroids(40).to_csv('asteroids.csv', index=False)